Imports importantes

In [1]:
import pandas as pd
import numpy as np

Carregando Dataset

In [2]:
tmdb_df = pd.read_csv("tmdb_new.csv")
tmdb_df.head()
tmdb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12020 entries, 0 to 12019
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       12020 non-null  int64  
 1   title                    12020 non-null  object 
 2   vote_average             12020 non-null  float64
 3   vote_count               12020 non-null  float64
 4   release_date             12020 non-null  object 
 5   revenue                  12020 non-null  float64
 6   runtime                  12020 non-null  float64
 7   budget                   12020 non-null  float64
 8   original_language        12020 non-null  object 
 9   popularity               12020 non-null  float64
 10  genres                   12020 non-null  object 
 11  production_companies     11618 non-null  object 
 12  production_countries     11769 non-null  object 
 13  cast                     11970 non-null  object 
 14  director              

In [3]:
from scipy.sparse import save_npz
from scipy.sparse import csr_matrix
from collections import Counter


def str_para_categorico(df: pd.DataFrame, coluna: str, k: int) -> pd.DataFrame:
    """
    Converte atributo de string para matriz esparsa
    """
    c = Counter()
    for string in df[coluna]:
        if string is not np.nan:
            c.update(string.split(", ")) 
    res = pd.DataFrame(index=df["title"])
    
    n_cols = 0
    for i in c.most_common():
        n_cols += 1
        if i[1] < k:
            break
    print(coluna, n_cols, "colunas")

    for i in c.most_common():
        if i[1] < k:
            break
        nova_col = []
        for string in df[coluna]:
            if string is not np.nan:
                nova_col.append(1 if i[0] in string.split(", ") else 0)
            else:
                nova_col.append(0)
        res[i[0]] = np.array(nova_col, dtype=np.int8)
        res = res.copy()

    sparse = csr_matrix(res) 
    save_npz(f"matrizes_esparsas/sparse_{coluna}_{k}.npz", sparse) # Salva em forma de matriz esparsa scipy

    return res

In [4]:
def str_para_categorico_v2(df: pd.DataFrame, coluna: str, k: int) -> pd.DataFrame:
    """
    Converte atributo de string para matriz esparsa
    """
    c = Counter()
    for string in df[coluna]:
        if string is not np.nan:
            c.update(string.split(", ")) 
    res = pd.DataFrame(index=df["title"])
    
    n_cols = 0
    for i in c.most_common():
        n_cols += 1
        if i[1] < k:
            break
    print(coluna, n_cols, "colunas")

    for i in c.most_common():
        if i[1] < k:
            break
        nova_col = []
        for string in df[coluna]:
            if string is not np.nan:
                nova_col.append(1 if i[0] in string.split(", ") else 0)
            else:
                nova_col.append(0)
        res[i[0]] = np.array(nova_col, dtype=np.int8)
        res = res.copy()

    sparse = csr_matrix(res) 
    save_npz(f"matrizes_esparsas/sparse_{coluna}_{k}.npz", sparse) # Salva em forma de matriz esparsa scipy

    return res

Criar e salvar matrizes esparsas

In [5]:
for col in ["genres", "writers", "director", "director_of_photography", "music_composer", "cast", "production_countries", "original_language", "producers", "production_companies"]:
    try:
        str_para_categorico(tmdb_df, col, 25)
    except:
        print(col)

genres 19 colunas
writers 13 colunas
director 9 colunas
director_of_photography 46 colunas
music_composer 68 colunas
cast 751 colunas
production_countries 48 colunas
original_language 28 colunas
producers 162 colunas
production_companies 151 colunas


In [5]:
from scipy.sparse import load_npz
from scipy.sparse import hstack
import os

def abrir_matrizes_esparsas(k: int = 5):
    """
    Carrega e concatena matrizes esparsas
    """
    l = []
    for arquivo in os.listdir("./matrizes_esparsas"):
        try:
            k_ok = int(arquivo.split("_")[-1][:-4]) == k
        except:
            k_ok = False
        if arquivo[-4:] == ".npz" and k_ok:
            l.append(load_npz(f"./matrizes_esparsas/{arquivo}"))
            print(arquivo)
    return hstack(l)

In [6]:
sparse = abrir_matrizes_esparsas(25)

sparse_genres_25.npz
sparse_producers_25.npz
sparse_production_countries_25.npz
sparse_music_composer_25.npz
sparse_director_of_photography_25.npz
sparse_director_25.npz
sparse_production_companies_25.npz
sparse_writers_25.npz
sparse_cast_25.npz
sparse_original_language_25.npz


Separando Conjunto de treinamento e teste

In [7]:
from sklearn.model_selection import train_test_split
X_treino, X_teste, y_treino, y_teste = train_test_split(
    hstack([csr_matrix(tmdb_df[["budget", "runtime"]]), sparse]), tmdb_df["revenue"].to_numpy(), test_size=0.2, random_state=42
    )

In [9]:
from sklearn.ensemble import RandomForestRegressor
# from sklearn.linear_model import LinearRegression
# from sklearn.neural_network import MLPRegressor

l = RandomForestRegressor(random_state=42)

# forest ≃ 94_000_000
# votingRegressor ≃ 95_000_000
# mlp ≃ 101_000_000
# linear ≃ 102_000_000

l.fit(X_treino, y_treino)


RandomForestRegressor(random_state=42)

In [10]:
from sklearn.metrics import root_mean_squared_error

print(f"Erro médio no conjunto de treino: {root_mean_squared_error(y_treino, l.predict(X_treino))}")
print(f"Erro médio no conjunto de teste: {root_mean_squared_error(y_teste, l.predict(X_teste))}")

Erro médio no conjunto de treino: 35994938.63645456
Erro médio no conjunto de teste: 95965871.44893236


In [11]:
# Salvando modelo

import pickle

with open("modelos/random_forest_revenue_k=25.pkl", "wb") as f:
    pickle.dump(l, f)

In [13]:
# Carregando modelo

with open("modelos/random_forest_revenue_k=25.pkl", "rb") as f:
    l = pickle.load(f)

Testando diferentes entradas

In [ ]:
for col in ["genres", "writers", "director", "director_of_photography", "music_composer", "cast", "production_countries", "original_language", "producers", "production_companies"]:
    try:
        str_para_categorico(tmdb_df, col, 25)
    except:
        print(col)

19 colunas
243 colunas
146 colunas
258 colunas
219 colunas
4782 colunas
60 colunas
40 colunas
948 colunas
539 colunas


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

for col in ["genres", "writers", "director", "director_of_photography", "music_composer", "cast", "production_countries", "original_language", "producers", "production_companies"]:
        
    model = RandomForestRegressor(random_state=42)
    X_treino, y_treino = load_npz(f"matrizes_esparsas/sparse_{col}_25.npz"), tmdb_df["revenue"].to_numpy()

    scores = cross_validate(model, X_treino, y_treino, scoring="neg_root_mean_squared_error")

    print(f"{col:<25}: {-np.mean(scores["test_score"]):9.6f}")

genres                   : 138723107.6824011
writers                  : 142197433.7784055
director                 : 147222688.06148905
director_of_photography  : 145851720.00237873
music_composer           : 137214865.64039096
cast                     : 128283525.80804142
production_countries     : 145943117.21179718
original_language        : 146263991.9030364
producers                : 131070716.72979884
production_companies     : 130899912.29972067


In [27]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

for col in ["genres", "writers", "director", "director_of_photography", "music_composer", "cast", "production_countries", "original_language", "producers", "production_companies"]:
        
    model = RandomForestRegressor(random_state=42)
    X_treino, y_treino = hstack([load_npz(f"matrizes_esparsas/sparse_{col}_25.npz"), csr_matrix(tmdb_df[["budget", "runtime"]])]), tmdb_df["revenue"].to_numpy()

    scores = cross_validate(model, X_treino, y_treino, scoring="neg_root_mean_squared_error")

    print(f"{col:<25}: {-np.mean(scores["test_score"]):09.6f}")

genres                   : 105508610.967504
writers                  : 114043198.834922
director                 : 114814977.169206
director_of_photography  : 115516415.311278
music_composer           : 110415274.403900
cast                     : 105486447.680854
production_countries     : 114050035.046966
original_language        : 114952506.036253
producers                : 111440826.193243
production_companies     : 109668051.599942


Prevendo Rating

In [11]:
from sklearn.model_selection import train_test_split
X_treino, X_teste, y_treino, y_teste = train_test_split(
    hstack([csr_matrix(tmdb_df[["budget", "runtime"]]), sparse]), tmdb_df["revenue"].to_numpy(), test_size=0.2, random_state=42
    )

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    hstack([csr_matrix(tmdb_df[["budget", "runtime"]]), sparse]), tmdb_df["revenue"].to_numpy(), test_size=0.2, random_state=42
    )

forest = RandomForestRegressor(random_state=42)
linear = LinearRegression()
mlp = MLPRegressor(random_state=42)
kneighbor = KNeighborsRegressor()

scores_forest = cross_validate(forest, X_treino, y_treino, scoring="neg_root_mean_squared_error")
scores_linear = cross_validate(linear, X_treino, y_treino, scoring="neg_root_mean_squared_error")
scores_mlp = cross_validate(mlp, X_treino, y_treino, scoring="neg_root_mean_squared_error")
scores_neighbor = cross_validate(kneighbor, X_treino, y_treino, scoring="neg_root_mean_squared_error")

print(f"forest   : {-np.mean(scores_forest["test_score"]):9.6f}")
print(f"linear   : {-np.mean(scores_linear["test_score"]):9.6f}")
print(f"neighbor : {-np.mean(scores_neighbor["test_score"]):9.6f}")
print(f"mlp      : {-np.mean(scores_mlp["test_score"]):9.6f}")


forest   : 94885413.331245
linear   : 102315185.795365
neighbor : 112382151.998673
mlp      : 102382621.100275


In [12]:
from sklearn.metrics import root_mean_squared_error

models = [
    RandomForestRegressor(random_state=42),
    LinearRegression(),
    MLPRegressor(random_state=42),
    KNeighborsRegressor()
]
models_names = ["forest", "linear", "neighbor", "mlp"]

for m, name in zip(models, models_names):
    m.fit(X_treino, y_treino)
    print(f"Erro médio no conjunto de treino {name}: {root_mean_squared_error(y_treino, m.predict(X_treino))}")
    print(f"Erro médio no conjunto de teste {name}: {root_mean_squared_error(y_teste, m.predict(X_teste))}")

Erro médio no conjunto de treino forest: 35994938.63645456
Erro médio no conjunto de teste forest: 95965871.44893236
Erro médio no conjunto de treino linear: 102450046.07775152
Erro médio no conjunto de teste linear: 104192874.89457887
Erro médio no conjunto de treino neighbor: 102533505.23678614
Erro médio no conjunto de teste neighbor: 104302668.31296827
Erro médio no conjunto de treino mlp: 92655593.16669028
Erro médio no conjunto de teste mlp: 108162363.56570825
